# 02 · ResNet-50 baseline, FGSM and PGD (single image)

Exploratory sanity check on one image. The reusable versions of the model wrapper and attacks now live in `src/awsa/` (`models.py`, `attacks.py`).

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cpu")

print("Using device:", device)

In [ ]:
weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)
model = model.to(device)
model.eval()

print("ResNet-50 loaded successfully.")

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"

response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

display(image)

In [ ]:
preprocess = weights.transforms()

input_tensor = preprocess(image)
input_batch = input_tensor.unsqueeze(0).to(device)

print("Input shape:", input_batch.shape)

In [ ]:
with torch.no_grad():
    output = model(input_batch)

probabilities = torch.nn.functional.softmax(output[0], dim=0)

top_prob, top_class = torch.topk(probabilities, 5)

categories = weights.meta["categories"]

for i in range(5):
    class_name = categories[top_class[i]]
    probability = top_prob[i].item()
    print(f"{class_name}: {probability:.4f}")

In [ ]:
clean_pred_idx = top_class[0].item()
clean_pred_name = categories[clean_pred_idx]
clean_confidence = top_prob[0].item()

print("Clean prediction:", clean_pred_name)
print("Clean confidence:", clean_confidence)

## ⚠️ Superseded: FGSM in normalized space

The next two cells apply ε to the *normalized* ResNet input, so ε is not measured in pixel units and the result is not clamped to [0, 1]. Kept for the record only; the corrected pixel-space version follows.

In [ ]:
input_adv = input_batch.clone().detach()
input_adv.requires_grad = True

output = model(input_adv)

loss = torch.nn.functional.cross_entropy(
    output,
    torch.tensor([clean_pred_idx], device=device)
)

model.zero_grad()
loss.backward()

epsilon = 2/255

perturbation = epsilon * input_adv.grad.sign()

adversarial_image = input_adv + perturbation

In [ ]:
with torch.no_grad():
    adv_output = model(adversarial_image)

adv_probabilities = torch.nn.functional.softmax(adv_output[0], dim=0)

adv_top_prob, adv_top_class = torch.topk(adv_probabilities, 5)

for i in range(5):
    class_name = categories[adv_top_class[i]]
    probability = adv_top_prob[i].item()
    print(f"{class_name}: {probability:.4f}")

## FGSM in pixel space (corrected)

ε is measured on [0, 1] pixels; normalization happens only right before the model.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# Convert the normalized ResNet input back to pixel range [0, 1]
pixel_image = (input_batch * std + mean).clamp(0, 1).detach()
pixel_image.requires_grad = True

def normalize_for_resnet(x):
    return (x - mean) / std

# Forward pass
output = model(normalize_for_resnet(pixel_image))

loss = torch.nn.functional.cross_entropy(
    output,
    torch.tensor([clean_pred_idx])
)

model.zero_grad()
loss.backward()

# True pixel-space epsilon
epsilon = 8 / 255

adv_pixel = pixel_image + epsilon * pixel_image.grad.sign()
adv_pixel = adv_pixel.clamp(0, 1).detach()

# Normalize again before giving it to ResNet
adversarial_image = normalize_for_resnet(adv_pixel)

print("Correct pixel-space FGSM created.")

In [ ]:
with torch.no_grad():
    adv_output = model(adversarial_image)

adv_probabilities = torch.nn.functional.softmax(adv_output[0], dim=0)

adv_top_prob, adv_top_class = torch.topk(adv_probabilities, 5)

for i in range(5):
    class_name = categories[adv_top_class[i]]
    probability = adv_top_prob[i].item()
    print(f"{class_name}: {probability:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Convert tensors from [1, 3, H, W] to [H, W, 3]
original_display = pixel_image.detach().squeeze(0).permute(1, 2, 0).numpy()
adv_display = adv_pixel.detach().squeeze(0).permute(1, 2, 0).numpy()

# Difference between the two
noise = adv_display - original_display

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(original_display)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(adv_display)
plt.title("FGSM Adversarial")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow((noise * 10 + 0.5).clip(0, 1))
plt.title("Perturbation ×10")
plt.axis("off")

plt.show()

## PGD in pixel space

ε = 2/255, step α = 0.5/255, 10 steps, no random start.

In [ ]:
# PGD attack in true pixel space

epsilon = 2 / 255
alpha = 0.5 / 255
num_steps = 10

original = pixel_image.detach().clone()

# Start from the clean image
adv_pixel_pgd = original.clone()

for step in range(num_steps):
    adv_pixel_pgd.requires_grad = True

    output = model(normalize_for_resnet(adv_pixel_pgd))

    loss = torch.nn.functional.cross_entropy(
        output,
        torch.tensor([clean_pred_idx])
    )

    model.zero_grad()
    loss.backward()

    # Small FGSM-like step
    adv_pixel_pgd = (
        adv_pixel_pgd
        + alpha * adv_pixel_pgd.grad.sign()
    ).detach()

    # Project back inside the allowed epsilon region
    delta = torch.clamp(
        adv_pixel_pgd - original,
        min=-epsilon,
        max=epsilon
    )

    adv_pixel_pgd = torch.clamp(
        original + delta,
        0,
        1
    ).detach()

print("PGD attack complete.")

In [ ]:
with torch.no_grad():
    pgd_output = model(normalize_for_resnet(adv_pixel_pgd))

pgd_probabilities = torch.nn.functional.softmax(pgd_output[0], dim=0)

pgd_top_prob, pgd_top_class = torch.topk(pgd_probabilities, 5)

for i in range(5):
    class_name = categories[pgd_top_class[i]]
    probability = pgd_top_prob[i].item()
    print(f"{class_name}: {probability:.4f}")

In [ ]:
pgd_display = adv_pixel_pgd.detach().squeeze(0).permute(1, 2, 0).numpy()
pgd_noise = pgd_display - original_display

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(original_display)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pgd_display)
plt.title("PGD Adversarial")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow((pgd_noise * 20 + 0.5).clip(0, 1))
plt.title("PGD Perturbation ×20")
plt.axis("off")

plt.show()